# E-Commerce Sales: Exploratory Data Analysis

**Author:** Blessing Osuagwu

This notebook explores a public, 1,000-row synthetic e-commerce transaction dataset. It answers questions about transaction value, category basket size, regional performance, monthly sales, shipping status, and returns. It does not estimate profit, customer lifetime value, or causal effects.

## Data source

Download `realistic_e_commerce_sales_data.csv` from the [Kaggle dataset](https://www.kaggle.com/datasets/brsahan/e-commerce-dataset/data?select=realistic_e_commerce_sales_data.csv) and place it in `data/`, or upload it to the Colab session. The CSV is not included in this repository. Check the dataset's license and terms before redistributing the file.

## 1. Setup and load the data

Install the packages listed in `requirements.txt` if they are not already available.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

DATA_FILE = Path("data/realistic_e_commerce_sales_data.csv")
if not DATA_FILE.exists():
    DATA_FILE = Path("/content/realistic_e_commerce_sales_data.csv")
if not DATA_FILE.exists():
    raise FileNotFoundError(
        "Download realistic_e_commerce_sales_data.csv from the Kaggle source linked "
        "in the README and place it in data/ or /content/."
    )

raw = pd.read_csv(DATA_FILE)
print(f"Rows: {len(raw):,} | Columns: {raw.shape[1]}")
raw.head()

In [ ]:
required = {
    "Customer ID", "Gender", "Region", "Age", "Product Name", "Category",
    "Unit Price", "Quantity", "Total Price", "Shipping Fee",
    "Shipping Status", "Order Date",
}
missing_columns = required - set(raw.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")

raw.info()
raw.isna().sum().sort_values(ascending=False)

## 2. Clean and define measures

The original notebook removed rows using sequential 99th-percentile cutoffs, which removed a large share of records and was not justified by the data. This version keeps valid transactions, reports missingness, and does not trim outliers by default. It treats each row as a transaction record because the dataset has no order ID. `Total Price` is treated as item sales; shipping is shown separately and added only for the approximate customer charge.

In [ ]:
df = raw.copy()

for col in ["Age", "Unit Price", "Quantity", "Total Price", "Shipping Fee"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")

rows_before = len(df)
df = df.dropna(subset=["Order Date", "Unit Price", "Quantity", "Total Price"])
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Region"] = df["Region"].fillna(df["Region"].mode().iat[0])
df["Shipping Status"] = df["Shipping Status"].fillna("Unknown").astype(str).str.strip().str.title()
df["Product Name"] = df["Product Name"].astype(str).str.strip().str.title()
df["Category"] = df["Category"].astype(str).str.strip().str.title()
df["Shipping Fee"] = df["Shipping Fee"].fillna(0)
df["Item Sales"] = df["Total Price"]
df["Customer Charge"] = df["Item Sales"] + df["Shipping Fee"]

price_mismatch = ~np.isclose(
    df["Total Price"], df["Unit Price"] * df["Quantity"], rtol=0, atol=0.01
)
print(f"Rows retained: {len(df):,} of {rows_before:,}")
print("Rows where Total Price differs from Unit Price × Quantity by more than $0.01:", int(price_mismatch.sum()))
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Status values:", sorted(df["Shipping Status"].unique()))
df[["Item Sales", "Shipping Fee", "Customer Charge", "Quantity"]].describe()

## 3. Business questions

### Q1. Which age group has the highest transaction value?

This compares totals in the available sample. It does not show that age causes higher spending.

In [ ]:
age_bins = [0, 18, 35, 50, 65, 120]
age_labels = ["0–18", "19–35", "36–50", "51–65", "66+"]
df["Age Group"] = pd.cut(df["Age"], bins=age_bins, labels=age_labels, include_lowest=True)
age_sales = df.groupby("Age Group", observed=False)["Item Sales"].sum().sort_values(ascending=False)
display(age_sales.rename("Item sales ($)"))
age_sales.plot(kind="bar", color="#4c78a8", title="Item sales by age group", ylabel="Item sales ($)", xlabel="Age group")
plt.tight_layout(); plt.show()

### Q2. Which categories have the largest average basket size?

Without an order ID, basket size is approximated as the mean quantity per row, not a verified multi-item basket.

In [ ]:
basket = df.groupby("Category")["Quantity"].mean().sort_values(ascending=False)
display(basket.rename("Mean units per transaction row"))
basket.sort_values().plot(kind="barh", color="#72b7b2", title="Mean units per transaction row", xlabel="Mean quantity")
plt.tight_layout(); plt.show()

### Q3. Do higher-priced categories sell fewer units?

There are only a few categories, so the chart is descriptive and should not be presented as evidence of a general price-demand relationship.

In [ ]:
category_summary = df.groupby("Category").agg(
    mean_unit_price=("Unit Price", "mean"), total_units=("Quantity", "sum"), transaction_rows=("Quantity", "size")
).sort_values("mean_unit_price")
display(category_summary)
sns.scatterplot(data=category_summary.reset_index(), x="mean_unit_price", y="total_units", hue="Category", s=120)
plt.title("Category price and units in this sample"); plt.xlabel("Mean unit price ($)"); plt.ylabel("Total units")
plt.tight_layout(); plt.show()

### Q4. How do sales differ by region?

Show both total sales and mean sales per transaction row so that volume and typical transaction value are not conflated.

In [ ]:
region_summary = df.groupby("Region").agg(
    item_sales=("Item Sales", "sum"), mean_item_sales=("Item Sales", "mean"), transaction_rows=("Item Sales", "size")
).sort_values("item_sales", ascending=False)
display(region_summary)
region_summary["item_sales"].plot(kind="bar", color="#f58518", title="Item sales by region", ylabel="Item sales ($)")
plt.tight_layout(); plt.show()

### Q5. When are sales highest?

Aggregate by year and month. If the dataset covers only one year, this shows monthly variation in that year, not a repeatable seasonal pattern.

In [ ]:
df["Month"] = df["Order Date"].dt.to_period("M").astype(str)
monthly_sales = df.groupby("Month")["Item Sales"].sum().sort_index()
display(monthly_sales.rename("Item sales ($)"))
monthly_sales.plot(marker="o", color="#54a24b", title="Monthly item sales", ylabel="Item sales ($)", xlabel="Month")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

### Q6. How do shipping fees and transaction values vary by shipping status?

This is descriptive. Shipping status may be associated with transaction value, but the analysis does not show that shipping status caused the difference.

In [ ]:
shipping_summary = df.groupby("Shipping Status").agg(
    transaction_rows=("Customer Charge", "size"),
    mean_shipping_fee=("Shipping Fee", "mean"),
    mean_customer_charge=("Customer Charge", "mean"),
).sort_values("mean_customer_charge", ascending=False)
display(shipping_summary)
sns.boxplot(data=df, x="Shipping Status", y="Customer Charge", color="#b279a2")
plt.title("Customer charge by shipping status"); plt.xticks(rotation=25); plt.tight_layout(); plt.show()

### Q7. Which products have the most returned transaction rows?

Counts alone do not account for how often each product was sold. The rate table below divides returned rows by all observed rows for each product.

In [ ]:
is_returned = df["Shipping Status"].eq("Returned")
product_returns = df.assign(returned=is_returned).groupby("Product Name").agg(
    returned_rows=("returned", "sum"), transaction_rows=("returned", "size")
)
product_returns["return_rate"] = product_returns["returned_rows"] / product_returns["transaction_rows"]
display(product_returns.sort_values("returned_rows", ascending=False).head(10))
product_returns.sort_values("returned_rows").tail(10)["returned_rows"].plot(kind="barh", color="#e45756", title="Products with most returned transaction rows")
plt.tight_layout(); plt.show()

### Q8. Which regions have the highest observed return-row rate?

The denominator is transaction rows, not unique orders, because the dataset has no order ID.

In [ ]:
region_returns = df.assign(returned=is_returned).groupby("Region").agg(
    returned_rows=("returned", "sum"), transaction_rows=("returned", "size")
)
region_returns["return_rate"] = region_returns["returned_rows"] / region_returns["transaction_rows"]
display(region_returns.sort_values("return_rate", ascending=False))
region_returns["return_rate"].sort_values().mul(100).plot(kind="barh", color="#e45756", title="Returned transaction rows by region", xlabel="Rate (%)")
plt.tight_layout(); plt.show()

## 4. Interpretation and next steps

Run the notebook from top to bottom and use the displayed tables and charts to write a short findings section. Avoid causal language: this is a small synthetic dataset, and regional or demographic differences may reflect the way the data was generated. Add a dashboard only after the analysis and metric definitions are stable.